# Phase 2 Exploratory Data Analysis

Selected city: Stockholm, Sweden

Data source: Inside Airbnb

This notebook is a readable companion to `src/run_phase2_eda.py`. The script is the reproducible source for the generated charts and report.

## Important Dataset Limitation

The Stockholm calendar dataset contains 100% missing values for `price` and `adjusted_price` in the raw source file. Therefore, calendar-based pricing analysis such as monthly price trends and weekday vs weekend price comparison cannot be performed using calendar data.

Calendar data is used for availability analysis only. Pricing analysis uses `listings_clean.price`.

In [ ]:
from pathlib import Path
import sqlite3

import pandas as pd

project_root = Path.cwd()
db_path = project_root / 'data' / 'processed' / 'airbnb_stockholm.db'
figures_dir = project_root / 'reports' / 'figures'

db_path

In [ ]:
with sqlite3.connect(db_path) as con:
    listings = pd.read_sql_query(
        '''
        SELECT id, host_id, host_name, neighbourhood_cleansed, property_type,
               room_type, price, availability_365, number_of_reviews,
               review_scores_rating
        FROM listings_clean
        ''',
        con,
    )
    calendar = pd.read_sql_query(
        '''
        SELECT listing_id, date, available, minimum_nights, maximum_nights
        FROM calendar_clean
        ''',
        con,
        parse_dates=['date'],
    )
    reviews = pd.read_sql_query(
        'SELECT date FROM reviews_clean WHERE date IS NOT NULL',
        con,
        parse_dates=['date'],
    )

listings.shape, calendar.shape, reviews.shape

## Key Summary Metrics

In [ ]:
summary = {
    'total_listings': len(listings),
    'total_hosts': listings['host_id'].nunique(),
    'total_neighbourhoods': listings['neighbourhood_cleansed'].nunique(),
    'median_listing_price': listings['price'].median(),
    'average_review_score': listings['review_scores_rating'].mean(),
    'available_calendar_days_pct': calendar['available'].mean() * 100,
    'listings_missing_price_pct': listings['price'].isna().mean() * 100,
}

pd.Series(summary).round(2)

## Listings and Room Types

Business interpretation: room type is the first major segmentation lens. It helps avoid comparing fundamentally different products, such as private rooms and entire homes.

![Listings by room type](../reports/figures/listings_by_room_type.png)

![Median price by room type](../reports/figures/median_price_by_room_type.png)

## Price Analysis From Listings

Business interpretation: pricing should use `listings_clean.price`. The price distribution chart is capped at the 99th percentile for visualization only; source data is not modified.

![Listing price distribution](../reports/figures/listing_price_distribution_capped.png)

![Median price by neighbourhood](../reports/figures/median_price_top_neighbourhoods.png)

![Median price by property type](../reports/figures/median_price_by_property_type.png)

## Availability Analysis From Calendar

Business interpretation: availability can show open supply, but it should not be treated as confirmed occupancy. Unavailable dates may be booked, blocked, or inactive.

![Availability counts](../reports/figures/availability_counts.png)

![Availability by month](../reports/figures/availability_by_month.png)

![Availability by room type](../reports/figures/availability_by_room_type.png)

## Host Analysis

Business interpretation: host portfolio size helps distinguish occasional hosts from more professional operators.

![Top hosts by listings](../reports/figures/top_hosts_by_listings.png)

![Host segment price](../reports/figures/host_segment_price.png)

## Review Analysis

Business interpretation: reviews are useful quality and activity signals, but review volume is not the same as booking volume.

![Review score distribution](../reports/figures/review_score_distribution.png)

![Price vs reviews](../reports/figures/price_vs_reviews.png)

![Top neighbourhood review scores](../reports/figures/top_neighbourhood_review_scores.png)

![Reviews by month](../reports/figures/reviews_by_month.png)

## Reproducible Command

Run the Phase 2 EDA script from the project root:

```bash
python src/run_phase2_eda.py
```